# Fine-Tuning Pretrained Models

<a target="_blank" href="https://colab.research.google.com/github/imamitjain/notebooks/blob/main/04-llm-and-transformers/03_fine_tuning_models.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objective:** Adapt a pretrained transformer to a custom task — data preparation, training with Hugging Face Trainer, evaluation, and saving/loading models.

**Prerequisites:** Hugging Face pipelines (notebook 02)

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q numpy torch transformers datasets accelerate


In [ ]:
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          TrainingArguments, Trainer, pipeline)
from datasets import load_dataset
import numpy as np
import torch

## 1. Loading and Preparing a Dataset

In [ ]:
dataset = load_dataset("imdb", split={"train": "train[:2000]", "test": "test[:500]"})

print(f"Train: {len(dataset['train'])} samples")
print(f"Test:  {len(dataset['test'])} samples")
print(f"\nSample:")
print(f"  Label: {dataset['train'][0]['label']} ({'positive' if dataset['train'][0]['label'] == 1 else 'negative'})")
print(f"  Text:  {dataset['train'][0]['text'][:200]}...")

## 2. Tokenizing Data for the Model

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=256)

tokenized = dataset.map(tokenize_function, batched=True)

tokenized["train"].set_format("torch", columns=["input_ids", "attention_mask", "label"])
tokenized["test"].set_format("torch", columns=["input_ids", "attention_mask", "label"])

print(f"Tokenized sample keys: {list(tokenized['train'][0].keys())}")
print(f"Input IDs shape: {tokenized['train'][0]['input_ids'].shape}")

## 3. Setting Up the Trainer

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=50,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()
    return {"accuracy": accuracy}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    compute_metrics=compute_metrics,
)

print(f"Model: {model_name}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable:  {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 4. Training and Monitoring Loss

In [ ]:
trainer.train()

## 5. Evaluation and Metrics

In [ ]:
results = trainer.evaluate()
print("Evaluation Results:")
for key, value in results.items():
    print(f"  {key}: {value:.4f}" if isinstance(value, float) else f"  {key}: {value}")

## 6. Saving, Loading, and Inference

In [ ]:
model.save_pretrained("./my_model")
tokenizer.save_pretrained("./my_model")

# Load and use for inference
inference_pipe = pipeline("sentiment-analysis", model="./my_model", tokenizer="./my_model")

test_texts = [
    "This movie was fantastic! Great acting and storyline.",
    "Boring and predictable. A waste of time.",
    "It had some good moments but overall was disappointing.",
]

for text in test_texts:
    result = inference_pipe(text)[0]
    label = "Positive" if result["label"] == "LABEL_1" else "Negative"
    print(f"  '{text[:50]}...' → {label} ({result['score']:.4f})")

## Try It Yourself

1. Fine-tune a DistilBERT model on the AG News dataset for topic classification. Report accuracy per class.
2. Experiment with learning rates [1e-5, 2e-5, 5e-5] and plot validation loss for each. Which works best?
3. Fine-tune a smaller model (e.g., `prajjwal1/bert-tiny`) and compare speed vs accuracy tradeoffs against DistilBERT.

In [ ]:
# Your code here